# SpaceX Falcon 9 – Interactive Visual Analytics with Folium
Mark all launch sites, the success/failed launches per site and the distances from a launch site to its proximities.

In [1]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

spacex_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


## Task 1: Mark all launch sites on a map

In [2]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    folium.Circle(coordinate, radius=20000, color='#d35400', fill=True).add_child(
        folium.Popup(site['Launch Site'])).add_to(site_map)
    folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 16px; color:#d35400; white-space: nowrap;"><b>%s</b></div>' % site['Launch Site'])).add_to(site_map)

site_map.save('images/map_sites.html')
site_map

## Task 2: Mark the success/failed launches for each site

In [3]:
marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')

site_map = folium.Map(location=[28.573255, -80.646895], zoom_start=17)
site_map.add_child(marker_cluster)
for _, record in spacex_df.iterrows():
    folium.Marker(location=[record['Lat'], record['Long']],
                  icon=folium.Icon(color='white', icon_color=record['marker_color']),
                  popup=record['Launch Site']).add_to(marker_cluster)

site_map.save('images/map_outcomes.html')
site_map

## Task 3: Calculate the distances between a launch site and its proximities

In [4]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

launch_site = launch_sites_df[launch_sites_df['Launch Site'] == 'CCAFS SLC-40'].iloc[0]
site_lat, site_lon = launch_site['Lat'], launch_site['Long']

# Proximity coordinates picked with the MousePosition plugin
proximities = {
    'Coastline':            (28.56342, -80.56794),
    'Railway':              (28.57206, -80.58526),
    'Highway':              (28.56329, -80.57076),
    'City (Titusville)':    (28.61208, -80.80764),
}
# Label offsets (pixels) so the labels do not overlap each other
label_anchor = {'Coastline': (-10, -12), 'Railway': (170, 20), 'Highway': (-10, 30), 'City (Titusville)': (0, 0)}

site_map = folium.Map(location=[site_lat + 0.002, site_lon + 0.004], zoom_start=15)
site_map.add_child(MousePosition(position='topright', separator=' Long: ', prefix='Lat:'))
folium.Marker([site_lat, site_lon], popup='CCAFS SLC-40',
              icon=folium.Icon(color='blue', icon='rocket', prefix='fa')).add_to(site_map)

distances = {}
for name, (lat, lon) in proximities.items():
    d = calculate_distance(site_lat, site_lon, lat, lon)
    distances[name] = round(d, 2)
    folium.Marker([lat, lon], icon=DivIcon(icon_size=(160, 20), icon_anchor=label_anchor[name],
        html='<div style="font-size: 13px; color:#d35400; background:white; padding:1px 3px;"><b>%s: %.2f KM</b></div>' % (name, d))
    ).add_to(site_map)
    folium.PolyLine(locations=[[site_lat, site_lon], [lat, lon]], weight=2, color='#1f5fbf').add_to(site_map)

site_map.save('images/map_proximity.html')
print(distances)
site_map

{'Coastline': 0.87, 'Railway': 1.29, 'Highway': 0.59, 'City (Titusville)': 23.19}
